# BRIDGE-JUPYTER-01 — PostgreSQL magics in Jupyter

## Goal

Use JupySQL to explore the disposable course PostgreSQL database from a notebook without embedding or displaying credentials. Keep SQL structure separate from data values, bound result size, and know when an application should use Psycopg directly instead.

**Level:** Intermediate/advanced  
**Stable lesson ID:** `bridge-jupyter-01`  
**Prerequisites:** Python Day 18, SQL Day 15, Bridge Day 3, and a reset disposable course database.

An IPython **line magic** begins with one `%` and consumes the rest of that line. A **cell magic** begins with `%%` and consumes the cell body. Here `%sql` is convenient for a short statement or a result assignment; `%%sql` keeps a multi-line query readable.

## Setup

Complete the repository setup outside this notebook. Start VS Code or Jupyter from a shell where `DS60_DATABASE_URL` is set to the disposable `advanced_sql_training` database. Do not paste that value into a cell, print it, or save it in notebook output. The companion guide has separate Windows PowerShell and macOS/Linux commands.

In [ ]:
# ruff: noqa: E501 -- JupySQL line magics stay on one physical line.
%load_ext sql

JupySQL accepts a SQLAlchemy engine. The next cell passes the environment value through the repository's shared course-target validator **before** SQLAlchemy sees it. The validator permits only the disposable `advanced_sql_training` database over a native local socket or loopback host (`localhost`, `127.0.0.1`, or `::1`). It rejects remote and multi-host authorities plus routing, service, file-reading, and unsupported query overrides. The cell then selects SQLAlchemy's Psycopg 3 dialect and creates a lazy engine. `create_engine()` does not print the URL and normally does not connect until first use.

In [ ]:
import os

from sqlalchemy import create_engine
from sqlalchemy.engine import make_url

from ds60sqlpy.sql_notebook import (
    COURSE_DATABASE_NAME,
    validate_course_database_target,
)

raw_database_url = os.environ.get("DS60_DATABASE_URL", "").strip()
if not raw_database_url:
    raise RuntimeError(
        "Set DS60_DATABASE_URL in the shell that starts this notebook, then restart the kernel."
    )

validated_database_target = validate_course_database_target(raw_database_url)
if validated_database_target == COURSE_DATABASE_NAME:
    validated_database_target = f"postgresql:///{COURSE_DATABASE_NAME}"

course_url = make_url(validated_database_target)
psycopg_url = course_url.set(drivername="postgresql+psycopg")
engine = create_engine(psycopg_url, pool_pre_ping=True)
assert engine.url.drivername == "postgresql+psycopg"
assert course_url.database == COURSE_DATABASE_NAME

Hide connection feedback before binding the engine. `autolimit` bounds rows fetched; `displaylimit` only shortens what is displayed and does not by itself protect memory. Keep `autopandas=False` initially so you can practise explicit conversion. Named `:value` parameters are enabled for safe value binding.

In [ ]:
%config SqlMagic.displaycon = False
%config SqlMagic.autolimit = 200
%config SqlMagic.displaylimit = 25
%config SqlMagic.autopandas = False
%config SqlMagic.named_parameters = "enabled"

In [ ]:
%sql engine --alias ds60-course

In [ ]:
%sql --connections

## Steps

### 1. Compare line and cell magics

Use a line magic for a small diagnostic. Use a cell magic when SQL should span multiple lines. These cells are read-only.

In [ ]:
%sql SELECT current_database() AS database_name, current_user AS database_user

In [ ]:
%%sql
SELECT customer_id, full_name, country, segment
FROM training.customers
ORDER BY customer_id
LIMIT 5;

### 2. Bind data values and convert a result

Python variables referenced as `:value` are sent through the database parameter boundary. They are data, not executable SQL. The query text keeps static table and column names.

In [ ]:
order_status = "paid"
minimum_total = 250

In [ ]:
orders_result = %sql SELECT order_id, customer_id, total_amount FROM training.orders WHERE status = :order_status AND total_amount >= :minimum_total ORDER BY total_amount DESC, order_id LIMIT 20

In [ ]:
orders_frame = orders_result.DataFrame()
assert list(orders_frame.columns) == ["order_id", "customer_id", "total_amount"]
orders_frame.head()

### 3. Use `autopandas` deliberately

With `autopandas=True`, an assigned `%sql` result is already a pandas DataFrame. Pandas controls display in this mode, so keep an explicit SQL `LIMIT` as well as `autolimit`.

In [ ]:
%config SqlMagic.autopandas = True
customer_frame = %sql SELECT customer_id, full_name, country FROM training.customers ORDER BY customer_id LIMIT 10
%config SqlMagic.autopandas = False

In [ ]:
assert customer_frame.shape[0] <= 10
assert {"customer_id", "full_name", "country"}.issubset(customer_frame.columns)

### 4. Separate binding from code generation

JupySQL also supports Jinja text such as `{{value}}`. That renders SQL source *before* the driver executes it, so an untrusted value can change SQL structure. Treat Jinja as reviewed code generation, not value binding. Prefer `:value` for data.

A parameter cannot represent an identifier: `FROM :table_name` is not valid identifier binding. Keep identifiers static in exploratory notebooks. In application code, validate a genuinely dynamic choice with an allowlist and compose it with `psycopg.sql.Identifier`.

In [ ]:
chosen_customer_id = 3
customer_orders = %sql SELECT order_id, order_date, status, total_amount FROM training.orders WHERE customer_id = :chosen_customer_id ORDER BY order_date, order_id LIMIT 25

### 5. Treat transactions as an explicit boundary

JupySQL's `SqlMagic.autocommit` defaults to true. A notebook is easy to run out of order, so this lesson stays read-only. For a multi-statement write, use a reviewed `engine.begin()` block or Psycopg transaction context and make commit/rollback ownership visible. Do not assume toggling a magic setting gives an application a complete retry-safe unit of work.

In [ ]:
from sqlalchemy import text

with engine.begin() as connection:
    observed_database = connection.scalar(text("SELECT current_database()"))

assert observed_database == "advanced_sql_training"

### 6. Know when to leave magics

Use magics for interactive, bounded exploration. Prefer Psycopg application code for reusable functions, typed row mapping, COPY/streaming, explicit transactions, retry classification, pooling, async work, cancellation, structured logs, and fake-backed tests. A notebook can call those tested functions instead of becoming the only copy of production logic.

## Exercises

**Focus:** Use JupySQL for bounded, reviewable PostgreSQL exploration while keeping credentials out of cells and values separate from SQL structure.

**Assumptions:** The notebook reads `DS60_DATABASE_URL`, binds an explicit SQLAlchemy `postgresql+psycopg` engine, disables connection display, and keeps live cells tagged.

**Failure to watch for:** Notebook output, connection displays, Jinja rendering, or unbounded result materialization can leak secrets or turn exploration into unsafe application behavior.

Complete these in order. Keep this notebook answer-free, use bounded
read-only SQL, and record which checks are structural versus live.

1. **Magic selection:** Run the line diagnostic and multi-line training query; explain why one
   `%sql` form is easier to review for each statement.
   - **Hint:** Choose from statement shape and reviewability, not from different security
     semantics.
   - **Verify:** Run `%sql SELECT current_database()` and the formatted customer cell query; record the same database name plus bounded rows, then explain that line magic suits a short diagnostic while cell magic exposes multi-line structure.
2. **Security testing:** Bind an injection-shaped string to a harmless read-only comparison,
   verify it behaves as data, then remove the value.
   - **Hint:** Use `:name`; never paste the sentinel into SQL or notebook output.
   - **Verify:** Bind `US' OR TRUE --` through `:injection_value`; assert the harmless comparison returns zero/expected rows, no schema changes, and saved SQL still contains `:injection_value` rather than the rendered sentinel.
3. **SQL practice:** Query US customers whose lifetime order total meets a Python threshold;
   bind country and threshold, order deterministically, and limit to 10.
   - **Hint:** Aggregate after a left join, apply the threshold after grouping, and add customer
     ID as tie-breaker.
   - **Verify:** Assert the query keeps `:exercise_country` and `:exercise_minimum_total`, returns at most 10 US customers meeting the threshold, and orders equal totals by customer ID.
4. **DataFrame boundary:** Convert the assigned result with `.DataFrame()` and assert the
   expected columns in order.
   - **Hint:** Treat conversion as an explicit boundary and validate shape before analysis.
   - **Verify:** Convert the assigned result and assert columns equal `['customer_id', 'full_name', 'lifetime_total']` in that order, row count is at most 10, and `lifetime_total` is numeric/Decimal-compatible.
5. **Configuration:** Repeat one small query with `autopandas=True`, identify the changed return
   type, then restore it to false.
   - **Hint:** Notebook-global magic configuration is hidden state unless restored visibly.
   - **Verify:** With `autopandas=False`, record a JupySQL result supporting `.DataFrame()`; with it true, record a pandas DataFrame; finally inspect configuration and assert it is restored false.
6. **Memory reasoning:** Explain why `displaylimit=25` does not bound memory and identify the
   setting/query clause that does.
   - **Hint:** Rendering fewer rows is different from fetching fewer rows.
   - **Verify:** State that `displaylimit=25` changes rendering only; inspect `autolimit` and explicit SQL `LIMIT` as fetch bounds, and identify aggregation/server work that `LIMIT` does not bound.
7. **Identifier boundary:** Explain why `FROM :table_name` is not identifier binding and how a
   Psycopg application handles a validated dynamic identifier.
   - **Hint:** Bound parameters represent data values, never SQL grammar.
   - **Verify:** Show `FROM :table_name` fails as grammar rather than selecting a table; the application design must allowlist the name and compose it with `psycopg.sql.Identifier`.
8. **Architecture decision:** Write a decision note choosing interactive exploration or Psycopg
   application code using at least three criteria.
   - **Hint:** Consider reuse, transaction ownership, tests, dynamic structure, scale, and
     operational observability.
   - **Verify:** Choose notebook magics or Psycopg in a decision table covering reuse, typing/testing, transaction ownership, result size, and interactivity; the choice must follow those facts.
9. **Cleanup:** List active aliases, close `ds60-course`, dispose the engine, and verify no
   connection literal or output remains saved.
   - **Hint:** Both JupySQL alias state and SQLAlchemy pool state need explicit cleanup.
   - **Verify:** Run `%sql --connections`, close alias `ds60-course`, dispose the engine, and assert the final saved notebook has no connection URL, rendered credential, or non-empty output.
10. **Setup review:** Trace how the environment URL becomes a SQLAlchemy engine without ever
   being displayed and identify every validation step.
   - **Hint:** Run the shared course-target validator before `make_url`; distinguish allowed
     local transports from remote, multi-host, and connection-override routes.
   - **Verify:** Trace `DS60_DATABASE_URL` through `validate_course_database_target`, optional literal-name normalization, `make_url`, `postgresql+psycopg`, `create_engine`, and `%sql engine`; name the local transports it accepts and the redirect controls it rejects, then assert no step prints or evaluates the URL/engine as the final cell expression.
11. **Prediction:** Predict the difference between a `%sql` line assignment and a `%%sql` cell
   when both return the same rows.
   - **Hint:** Compare Python assignment syntax, multi-line readability, and result access.
   - **Verify:** Predict and confirm that assigned `%sql` returns a result object, while `%%sql` owns the remaining cell and cannot be placed on the assignment's right side; both return the same rows.
12. **Capacity:** Design a query/result-size check for a table with millions of rows and explain
   what remains unbounded after `LIMIT 25`.
   - **Hint:** Bound output, scan scope, and server work separately.
   - **Verify:** For a million-row table, show a reviewed aggregate/count plus an explicit `LIMIT 25`; record that returned rows are bounded while scan, sort, and server execution may remain large.
13. **Type binding:** Bind a `Decimal`, date, boolean, and list value in small read-only queries
   and record their PostgreSQL result types.
   - **Hint:** Let SQLAlchemy/JupySQL adapt Python values; do not pre-render literals.
   - **Verify:** Bind a `Decimal`, `date`, Boolean, and Python list through named parameters; record PostgreSQL types/values and assert none of their literal representations was pasted into SQL.
14. **Transaction reasoning:** Use `engine.begin()` for a rollback-safe teaching write in a
   disposable temporary scope, then explain why magics are not the transaction owner.
   - **Hint:** The checked-in notebook remains read-only; describe or run writes only in an
     explicitly authorized disposable lab.
   - **Verify:** Use `engine.begin()` with a temporary/course-owned rollback-safe scope and inspect cleanup; state that the context, not `%sql`, owns commit/rollback and leave the checked-in notebook read-only.
15. **Jinja boundary:** Demonstrate conceptually why `{{value}}` is code generation rather than
   safe value binding, including a harmless fixed example.
   - **Hint:** Rendered text becomes SQL before the driver sees parameters.
   - **Verify:** Render one harmless fixed Jinja example as text and compare it with `:value`: assert Jinja changes SQL source before execution while the named parameter leaves source unchanged.
16. **Notebook hygiene:** Validate nbformat, stable IDs, kernel metadata, live/static tags,
   empty outputs, and the absence of package-install magics, shell installs, URLs, and
   destructive SQL.
   - **Hint:** Inspect the serialized artifact, not only the visible notebook UI.
   - **Verify:** Run notebook validation and assert nbformat/stable IDs/kernel/tags are valid, outputs and execution counts are clear, and no install magic, shell install, URL literal, or destructive SQL exists.
17. **Offline review:** Explain how a learner without a running PostgreSQL server can still
   review this module and which claims remain unexecuted.
   - **Hint:** Separate structural/offline evidence from live query evidence.
   - **Verify:** Offline, compile/inspect Python and SQL cells, verify tags/metadata/parameters/cleanup text, and explicitly label database name, returned rows, and driver adaptation as unexecuted claims.
18. **Handoff:** Extract the capstone query into a Psycopg function design with typed inputs, a
   small cursor Protocol, and a fake-backed test.
   - **Hint:** Carry over SQL and value semantics while changing the ownership/testing surface.
   - **Verify:** Specify a typed Psycopg function whose cursor Protocol records static SQL plus a two-value parameter tuple; assert a fake returns the same ordered customer result without JupySQL state.


## Checks

Use the prepared variables and diagnostic cell below for the capstone query after attempting the numbered exercises above.

- Confirm every result is bounded and deterministically ordered.
- Confirm every data value uses `:name` binding and identifiers remain static.
- Distinguish structural checks from live PostgreSQL observations.
- Clear outputs and close/dispose both connection layers before saving.

In [ ]:
exercise_country = "US"
exercise_minimum_total = 500

# Add your result assignment and bound aggregate query in the next cell.

In [ ]:
%%sql
-- Replace this diagnostic row with the aggregate query described above.
SELECT :exercise_country AS country_to_bind,
       :exercise_minimum_total AS minimum_to_bind;

Before continuing, verify that the notebook contains no URL literal or saved output; every exploratory query is bounded; values use `:name`; identifiers are static; and no write was issued. Run `%sql --connections` to inspect the active alias, then close it explicitly.

In [ ]:
%sql --close ds60-course

In [ ]:
engine.dispose()

## Next Steps

Compare behavior with the separate solution notebook only after attempting the check. Then return to Bridge Days 3–5 for production-safe Psycopg adapters and tests, or continue to BRIDGE-OPS-01 for migration delivery, readiness, and redacted observability.